<a href="https://colab.research.google.com/github/JimYao314/AI-TryON/blob/main/Yao's_comfyui_v_3_Flux_2_Klein_315.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1️⃣ 系統環境初始化與降級偵測
import torch
import os

print("🔍 [1/2] 正在檢查運算環境...")
USE_CPU = not torch.cuda.is_available()

if USE_CPU:
    print("⚠️ 【警告】未偵測到 GPU！系統自動降級為純 CPU 模式。")
else:
    print(f"✅ 成功偵測到 GPU: {torch.cuda.get_device_name(0)}")

os.environ['COMFY_USE_CPU'] = 'true' if USE_CPU else 'false'

print("⚡ [2/2] 正在安裝極速下載神器 (aria2)...")
!apt-get -y install -qq aria2 > /dev/null
print("✅ 環境準備完畢！")

In [ ]:
# @title 2️⃣ Custom Nodes 模組化安裝區
import os
import subprocess

%cd /content
if not os.path.exists('/content/ComfyUI'):
    print("⏬ 正在 Clone ComfyUI 核心...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git > /dev/null

%cd /content/ComfyUI
print("📦 正在安裝 ComfyUI 核心依賴套件...")
!pip install -r requirements.txt -q

# 預先安裝底層庫 (避免 GGUF/ONNX 報錯)
print("📦 正在安裝底層運行庫 (ONNX, GGUF)...")
!pip install onnxruntime onnxruntime-gpu xformers -q

print("📦 正在同步 ComfyUI 自定義節點...")
NODE_DIR = "/content/ComfyUI/custom_nodes"
os.makedirs(NODE_DIR, exist_ok=True)

# 💡 未來只需在此陣列增刪 GitHub 連結
CUSTOM_NODES =[
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",
    # "https://github.com/yolain/ComfyUI-Easy-Sam3.git",
    # "https://github.com/city96/ComfyUI-GGUF.git",
    "https://github.com/rgthree/rgthree-comfy",
    # "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes",
    # "https://github.com/welltop-cn/ComfyUI-TeaCache",
]

for url in CUSTOM_NODES:
    repo_name = url.split('/')[-1].replace('.git', '')
    target_path = os.path.join(NODE_DIR, repo_name)

    if not os.path.exists(target_path):
        print(f"⬇️ 正在安裝: {repo_name}")
        subprocess.run(["git", "clone", url, target_path], check=True)
    else:
        print(f"⏩ 已存在，跳過: {repo_name}")

    # 🌟 新增：節點依賴自動安裝
    req_path = os.path.join(target_path, "requirements.txt")
    if os.path.exists(req_path):
        print(f"📦 安裝 {repo_name} 的專屬依賴...")
        # 隱藏冗長的下載日誌，但保留錯誤提示
        !pip install -r "{req_path}" -q

# 👑 終極防護：強制依賴鎖定 (Master Override)
# 無論上面的節點把版本改成什麼，最後一律強制對齊至最穩定的 1.x 終極版本，防止 SAM3 與其他節點衝突崩潰
print("🔨 正在執行底層依賴強制鎖定 (解決 Numpy/OpenCV 衝突)...")
!pip install numpy==1.26.4 opencv-python-headless==4.8.1.78 scikit-image -q 2>/dev/null
!pip install triton -q

print("✅ 節點與依賴同步完成！")

In [ ]:
# @title 3️⃣ 資源集中調度區 (處理空格檔名版)
import os
import subprocess
import urllib.request
from google.colab import userdata

# 🔐 讀取 Token (請確保 Colab Secrets 有 HF_Token 並開啟權限)
HF_TOKEN = userdata.get('HF_Token')
MY_REPO = "anadsf0314/my-sam3-weights"

def download_res(url, folder, filename):
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    if not os.path.exists(filepath):
        print(f"⏬ 正在下載: {filename} ...")
        header_cmd = f'--header="Authorization: Bearer {HF_TOKEN}"' if HF_TOKEN else ""
        safe_url = url.replace("/blob/", "/resolve/")
        # 使用 aria2c 高速下載模型
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {header_cmd} "{safe_url}" -d "{folder}" -o "{filename}"'
        subprocess.run(cmd, shell=True)
        if os.path.exists(filepath): print(f"✅ {filename} 下載完成")
    else:
        print(f"⏩ 已存在: {filename}")

# 📦 模型配置
MODEL_LIST =[
    {"url": f"https://huggingface.co/{MY_REPO}/resolve/main/flux-2-klein-9b-fp8.safetensors", "folder": "/content/ComfyUI/models/diffusion_models", "filename": "flux-2-klein-9b-fp8.safetensors"},
    {"url": f"https://huggingface.co/{MY_REPO}/resolve/main/qwen_3_8b_fp8mixed.safetensors", "folder": "/content/ComfyUI/models/text_encoders", "filename": "qwen_3_8b_fp8mixed.safetensors"},
    {"url": f"https://huggingface.co/{MY_REPO}/resolve/main/flux2-vae.safetensors", "folder": "/content/ComfyUI/models/vae", "filename": "flux2-vae.safetensors"},
    {"url": f"https://huggingface.co/{MY_REPO}/resolve/main/sam3-fp16.safetensors", "folder": "/content/ComfyUI/models/sam3", "filename": "sam3-fp16.safetensors"}
]

# 🗺️ 工作流配置 (已處理空格 %20)
WORKFLOW_LIST = [
    {
        # 🛠️ 關鍵修正：空格處改為 %20
        "url": f"https://huggingface.co/{MY_REPO}/resolve/main/Klein_v1.5%20GREAT.json",
        "path": "/content/ComfyUI/user/default/workflows/Klein_v1.5_GREAT.json"
    }
]

print("📥 開始同步資源...")
# 1. 下載模型
for m in MODEL_LIST: download_res(m["url"], m["folder"], m["filename"])

# 2. 下載工作流
for w in WORKFLOW_LIST:
    try:
        # 自動建立資料夾
        os.makedirs(os.path.dirname(w["path"]), exist_ok=True)

        req = urllib.request.Request(w["url"])
        if HF_TOKEN: req.add_header("Authorization", f"Bearer {HF_TOKEN}")

        with urllib.request.urlopen(req) as response, open(w["path"], 'wb') as out_file:
            out_file.write(response.read())
        print(f"✅ 工作流已備妥: {os.path.basename(w['path'])}")
    except Exception as e:
        print(f"❌ 工作流下載失敗。請檢查 HF 上的檔名是否完全吻合。錯誤: {e}")

print("🎉 所有資源調度完成！現在可以去啟動引擎了。")

In [ ]:
# @title 🛠️ 雲端工作流注入 - 診斷修復版
import urllib.request
import urllib.error
import os
from google.colab import userdata

# 1. 初始化設定
HF_TOKEN = userdata.get('HF_Token')
MY_REPO = "anadsf0314/my-sam3-weights"
UI_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(UI_DIR, exist_ok=True)

# 2. 定義清單 (確保空格已處理為 %20)
WORKFLOWS = [
    {
        "name": "🖥️ 前端 UI GREAT版",
        "url": f"https://huggingface.co/{MY_REPO}/resolve/main/Klein_v1.5%20GREAT.json",
        "path": os.path.join(UI_DIR, "Klein_v1.5_GREAT.json")
    },
    {
        "name": "🤖 後端 API 呼叫版",
        "url": f"https://huggingface.co/{MY_REPO}/resolve/main/Klein_v1.5_workflow_api.json",
        "path": "/content/ComfyUI/Klein_v1.5_workflow_api.json"
    }
]

def download_json(name, url, path):
    print(f"📡 正在嘗試連線至: {name}...") # 讓你知道程式有在動
    try:
        req = urllib.request.Request(url)
        if HF_TOKEN:
            req.add_header("Authorization", f"Bearer {HF_TOKEN}")

        # 加入 timeout=10，避免網路卡住時無限等待
        with urllib.request.urlopen(req, timeout=10) as response:
            with open(path, 'wb') as out_file:
                out_file.write(response.read())
        print(f"✅ 成功載入: {name}")

    except urllib.error.HTTPError as e:
        print(f"❌ 下載失敗 (HTTP {e.code}): {name}")
        if e.code == 404:
            print("   👉 原因：找不到檔案。請檢查 HF 上的檔名是否正確（大小寫、空格）。")
    except Exception as e:
        # 🛠️ 這裡修正了原本的 'p'，改為印出真實錯誤
        print(f"❌ 發生未預期錯誤: {type(e).__name__} - {e}")

# 3. 執行任務
print("📖 開始載入工作流任務...")
for wf in WORKFLOWS:
    download_json(wf["name"], wf["url"], wf["path"])

print("\n🎉 處理結束。")

In [ ]:
# @title 5️⃣ 啟動伺服器與固定 API 網址
import os
import time
import subprocess
from google.colab import userdata

# ==========================================
# 🔐 【設定區】
# ==========================================
NGROK_TOKEN = userdata.get("NGROK_TOKEN")
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")
# ==========================================

%cd /content/ComfyUI

print("🔧 正在以官方標準模式安裝 Ngrok (穩定性 100%)...")

# 1. 清理舊的殘留檔案
!rm -f ngrok-v3-stable-linux-amd64.tgz

# 2. 使用官方 APT 儲存庫安裝 (這能保證路徑正確且執行檔完整)
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list >/dev/null
!sudo apt-get update -qq && sudo apt-get install ngrok -y -qq >/dev/null

# 3. 設定 Authtoken
!ngrok config add-authtoken {NGROK_TOKEN} > /dev/null 2>&1

print("\n" + "🌟"*20)
print("🚀 【Ngrok 固定網址已啟動】")
print(f"👉 前端 UI 網址: https://{NGROK_DOMAIN}")
print(f"🤖 API 呼叫端點: https://{NGROK_DOMAIN}/prompt")
print("🌟"*20 + "\n")

# 4. 殺死舊的進程防止連線衝突
!pkill -f ngrok
time.sleep(1)

# 5. 核心修復：使用絕對路徑 '/usr/local/bin/ngrok' 並強制綁定 IPv4
try:
    subprocess.Popen(['ngrok', 'http', f'--domain={NGROK_DOMAIN}', '127.0.0.1:8188'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True # 確保獨立運行
    )
    print("📡 隧道連線中，準備啟動引擎...")
except Exception as e:
    print(f"❌ 隧道啟動失敗: {e}")

time.sleep(3) # 給予足夠時間建立隧道

# 6. 啟動 ComfyUI 引擎
print("🔥 正在啟動 ComfyUI 引擎 (即時日誌已開啟)...")
launch_cmd = "python main.py --use-pytorch-cross-attention --fp16-vae --fast"
if os.environ.get('COMFY_USE_CPU') == 'true':
    print("\n⚠️ 偵測到環境異常，以【純 CPU 模式】強制啟動。")
    launch_cmd += " --cpu"

!{launch_cmd}